# Библиотеки

In [4]:
from itertools import combinations
from pprint import pprint

import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split # разбивка на тестовую и обучающую выборку
from sklearn.metrics import classification_report, precision_recall_curve, roc_curve, roc_auc_score
from statsmodels.miscmodels.ordinal_model import OrderedModel

import seaborn as sns

## 11.6 Порядковая логистическая регрессия

- Пол, категориальная номинальная шкала: 1 = М, 2 = Ж
- Возраст, классическая интервальная шкала: в виде возраста в годах.
- Стаж, порядковая шкала: 0=нет, студент\учусь, не работал, 1=менее 5 лет, 2=до 10 лет, 3=до 20 лет, 4=более 20 лет
- Образование, порядковая шкала: 1 =среднее\ср.-спец, 2=высшее (вкл. несколько), 3=академическое (кандидат, доктор наук)
- ОтнПриезжие, порядковая шкала об отношении жителей центра одного города к "приезжим" (не коренным горожанам): 1=негативно, 2=нейтрально, 3=нормально (это будет наша зависимая переменная, которую мы будем пытаться прогнозировать\предсказывать переменными-предикторами перечисленными выше)

In [8]:
d = pd.read_excel('./data/Order.xlsx')

In [9]:
d.head()

,Пол,Возраст,Стаж,Образование,ОтнПриезжие
0,2,66,4,3,1
1,1,57,4,3,1
2,1,35,1,2,2
3,1,36,3,2,3
4,2,29,1,2,3


In [10]:
d = pd.get_dummies(d, columns=['Пол'], drop_first=True)

In [19]:
d.dtypes

Возраст        int64
Стаж           int64
Образование    int64
ОтнПриезжие    int64
Пол_2          int64
dtype: object

In [18]:
d['Пол_2'] = d['Пол_2'].apply(pd.to_numeric, errors='coerce')
d['Пол_2'] = d['Пол_2'].astype('int64')

In [13]:
d['Пол_2'].isna().sum()

np.int64(0)

In [21]:
m = OrderedModel(endog=d['ОтнПриезжие'], exog=d[['Пол_2', 'Возраст', 'Стаж', 'Образование']], distr='logit')

In [22]:
res=m.fit()

C:\pyenvs\da-python\Lib\site-packages\statsmodels\base\optimizer.py:748: RuntimeWarning: Maximum number of iterations has been exceeded.
  retvals = optimize.fmin(f, start_params, args=fargs, xtol=xtol,
C:\pyenvs\da-python\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [23]:
res = m.fit(maxiter=1000)

Optimization terminated successfully.
         Current function value: 1.070628
         Iterations: 633
         Function evaluations: 975


In [24]:
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OrderedModel Results                             
==============================================================================
Dep. Variable:            ОтнПриезжие   Log-Likelihood:                -321.19
Model:                   OrderedModel   AIC:                             654.4
Method:            Maximum Likelihood   BIC:                             676.6
Date:                Mon, 05 Jan 2026                                         
Time:                        22:13:14                                         
No. Observations:                 300                                         
Df Residuals:                     294                                         
Df Model:                           4                                         
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Пол_2           0.2565      0.217      1.181      0.238      -0.169       0.682
Возраст         0.0024      0.018      0.131      0.896      -0.033       0.038
Стаж           -0.2673      0.218     -1.224      0.221      -0.695       0.161
Образование     0.0682      0.332      0.206      0.837      -0.583       0.719
1/2            -1.0150      0.547     -1.857      0.063      -2.086       0.056
2/3             0.2030      0.097      2.096      0.036       0.013       0.393
===============================================================================
"""

In [27]:
d['pred'] = res.predict(d[['Пол_2', 'Возраст', 'Стаж', 'Образование']]).idxmax(axis=1) + 1

In [28]:
pd.crosstab(d['ОтнПриезжие'], d['pred'])

pred,1,3
ОтнПриезжие,,
1,45,50
2,26,60
3,34,85


In [29]:
d1 = pd.get_dummies(d, columns=['Стаж', 'Образование'], drop_first=True).astype('int64')
m1 = OrderedModel(endog=d1['ОтнПриезжие'], exog=d1[['Пол_2', 'Возраст', 'Стаж_1','Стаж_2','Стаж_3','Стаж_4','Образование_2','Образование_3']],distr='logit')
res1 = m1.fit(maxiter=1500)
res1.summary()

Optimization terminated successfully.
         Current function value: 1.066266
         Iterations: 1031
         Function evaluations: 1475


<class 'statsmodels.iolib.summary.Summary'>
"""
                             OrderedModel Results                             
==============================================================================
Dep. Variable:            ОтнПриезжие   Log-Likelihood:                -319.88
Model:                   OrderedModel   AIC:                             659.8
Method:            Maximum Likelihood   BIC:                             696.8
Date:                Mon, 05 Jan 2026                                         
Time:                        22:15:11                                         
No. Observations:                 300                                         
Df Residuals:                     290                                         
Df Model:                           8                                         
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Пол_2             0.3185      0.218      1.461      0.144      -0.109       0.746
Возраст          -0.0233      0.029     -0.815      0.415      -0.079       0.033
Стаж_1           -0.1347      0.413     -0.326      0.744      -0.944       0.674
Стаж_2           -0.3402      0.583     -0.584      0.559      -1.483       0.802
Стаж_3            0.1762      0.692      0.255      0.799      -1.180       1.532
Стаж_4           -0.1367      1.066     -0.128      0.898      -2.227       1.953
Образование_2     0.2658      0.494      0.538      0.591      -0.703       1.235
Образование_3    -0.3914      1.034     -0.378      0.705      -2.419       1.636
1/2              -1.4211      0.681     -2.087      0.037      -2.756      -0.086
2/3               0.2104      0.097      2.170      0.030       0.020       0.400
=================================================================================
"""